In [ ]:
%py
# Purpose: Comprehensive PySpark test suite for masking the last four digits of invoice_number in purgo_playground.d_product_revenue_clone table

import unittest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr
from pyspark.sql.types import StringType

class TestInvoiceNumberMasking(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Initialize SparkSession
        cls.spark = SparkSession.builder.appName("InvoiceNumberMaskingTests").getOrCreate()
        
        # Setup: Drop the clone table if it exists and create a fresh clone
        cls.clone_table = "purgo_playground.d_product_revenue_clone"
        cls.original_table = "purgo_playground.d_product_revenue"
        try:
            cls.spark.sql(f"DROP TABLE IF EXISTS {cls.clone_table}")
            cls.spark.sql(f"CREATE TABLE {cls.clone_table} AS SELECT * FROM {cls.original_table}")
        except Exception as e:
            # Handle failure to drop or clone table
            raise Exception(f"Setup failed: {e}")

    @classmethod
    def tearDownClass(cls):
        # Cleanup: Drop the clone table after tests
        try:
            cls.spark.sql(f"DROP TABLE IF EXISTS {cls.clone_table}")
            cls.spark.stop()
        except Exception as e:
            # Handle failure to drop table
            print(f"Cleanup failed: {e}")

    def test_mask_last_four_digits(self):
        """
        Test that the last four digits of invoice_number are masked correctly
        """
        # Execute masking logic
        try:
            df = self.spark.table(self.clone_table)
            masked_df = df.withColumn("invoice_number",
                                      expr("CASE WHEN invoice_number IS NOT NULL THEN CONCAT(SUBSTR(CAST(invoice_number AS STRING), 1, LENGTH(CAST(invoice_number AS STRING)) - 4), '****') ELSE NULL END"))
            masked_df.write.mode("overwrite").saveAsTable(self.clone_table)
        except Exception as e:
            self.fail(f"Masking logic execution failed: {e}")

        # Validate masking
        result_df = self.spark.table(self.clone_table).select("invoice_number").collect()
        for row in result_df:
            invoice = row.invoice_number
            if invoice is not None:
                self.assertTrue(invoice.endswith("****"), f"Invoice number {invoice} not masked correctly")
    
    def test_various_invoice_formats(self):
        """
        Test masking with various invoice_number formats
        """
        test_cases = {
            "1234567890": "123456****",
            "987654321": "98765****",
            "1000": "****",
            "56789": "5****"
        }
        for original, expected in test_cases.items():
            with self.subTest(original=original, expected=expected):
                try:
                    df = self.spark.table(self.clone_table).filter(col("invoice_number") == original)
                    masked_df = df.withColumn("masked_invoice",
                                              expr("CONCAT(SUBSTR(CAST(invoice_number AS STRING), 1, LENGTH(CAST(invoice_number AS STRING)) - 4), '****')"))
                    result = masked_df.select("masked_invoice").collect()
                    if result:
                        self.assertEqual(result[0].masked_invoice, expected)
                except Exception as e:
                    self.fail(f"Masking failed for invoice_number {original}: {e}")

    def test_non_numeric_invoice_number(self):
        """
        Test handling of non-numeric invoice_number values
        """
        try:
            df = self.spark.table(self.clone_table).filter(col("invoice_number") == "INV12345")
            masked_df = df.withColumn("invoice_number",
                                      expr("CONCAT(SUBSTR(invoice_number, 1, LENGTH(invoice_number) - 4), '****')"))
            masked_df.write.mode("overwrite").saveAsTable(self.clone_table)
            self.fail("Non-numeric invoice_number was masked without raising an error")
        except Exception as e:
            self.assertIn("Invalid invoice_number format for masking", str(e))

    def test_data_type_consistency(self):
        """
        Ensure invoice_number column is of type string after masking
        """
        df = self.spark.table(self.clone_table)
        schema = df.schema
        invoice_type = dict(schema)["invoice_number"].dataType
        self.assertEqual(invoice_type, StringType(), "invoice_number column is not of type string after masking")

    def test_missing_invoice_number_column(self):
        """
        Test behavior when invoice_number column is missing in clone table
        """
        try:
            # Drop invoice_number column
            df = self.spark.table(self.clone_table).drop("invoice_number")
            df.write.mode("overwrite").saveAsTable(self.clone_table)
            
            # Execute masking logic
            df = self.spark.table(self.clone_table)
            masked_df = df.withColumn("invoice_number",
                                      expr("CONCAT(SUBSTR(CAST(invoice_number AS STRING), 1, LENGTH(CAST(invoice_number AS STRING)) - 4), '****')"))
            masked_df.write.mode("overwrite").saveAsTable(self.clone_table)
            self.fail("Masking logic executed without invoice_number column")
        except Exception as e:
            self.assertIn("invoice_number column is missing in d_product_revenue_clone table", str(e))

    def test_column_count_validation(self):
        """
        Validate the number of columns in clone table matches the original table
        """
        original_df = self.spark.table(self.original_table)
        clone_df = self.spark.table(self.clone_table)
        self.assertEqual(len(original_df.columns), len(clone_df.columns), "Column count does not match between original and clone tables")

    def test_schema_validation(self):
        """
        Validate the schema of clone table matches the original table
        """
        original_schema = self.spark.table(self.original_table).schema
        clone_schema = self.spark.table(self.clone_table).schema
        self.assertEqual(original_schema, clone_schema, "Schemas do not match between original and clone tables")

    def test_null_handling(self):
        """
        Test that NULL invoice_number values remain NULL after masking
        """
        df = self.spark.table(self.clone_table).filter(col("invoice_number").isNull())
        masked_df = df.withColumn("invoice_number",
                                  expr("CASE WHEN invoice_number IS NOT NULL THEN CONCAT(SUBSTR(CAST(invoice_number AS STRING), 1, LENGTH(CAST(invoice_number AS STRING)) - 4), '****') ELSE NULL END"))
        masked_df.write.mode("overwrite").saveAsTable(self.clone_table)
        result_df = self.spark.table(self.clone_table).filter(col("invoice_number").isNull()).count()
        self.assertGreater(result_df, 0, "NULL invoice_number values were altered")

    def test_delta_lake_operations(self):
        """
        Test Delta Lake operations on the clone table
        """
        try:
            # Perform an update operation
            self.spark.sql(f"UPDATE {self.clone_table} SET is_returned = 1 WHERE product_id = 1001")
            updated_value = self.spark.table(self.clone_table).filter(col("product_id") == 1001).select("is_returned").collect()[0].is_returned
            self.assertEqual(updated_value, 1, "Delta Lake UPDATE operation failed")
            
            # Perform a delete operation
            self.spark.sql(f"DELETE FROM {self.clone_table} WHERE product_id = 1002")
            deleted_count = self.spark.table(self.clone_table).filter(col("product_id") == 1002).count()
            self.assertEqual(deleted_count, 0, "Delta Lake DELETE operation failed")
        except Exception as e:
            self.fail(f"Delta Lake operations failed: {e}")

    def test_cleanup_operations(self):
        """
        Ensure cleanup operations are performed correctly
        """
        try:
            # Drop the clone table
            self.spark.sql(f"DROP TABLE IF EXISTS {self.clone_table}")
            table_exists = self.spark.catalog.tableExists(self.clone_table)
            self.assertFalse(table_exists, "Clone table was not dropped successfully")
        except Exception as e:
            self.fail(f"Cleanup operation failed: {e}")

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)